<a href="https://colab.research.google.com/github/Amar234/agentic-AI-/blob/main/travel_assistent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

Travel Assistant AI

In [1]:

from warnings import filterwarnings
filterwarnings(action='ignore')

In [3]:
!pip install -U langchain langchain-core langchain-community langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.8
    Uninstalling langchain-text-splitters-0.3.8:
      Successfully uninstalled langchain-text-splitters-0.3.8
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.3.14
    Uninstalling langchain-community-0.3.14:
      Successfully uninstalled langchain-community-0.3.14
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.14
    Uninst

In [1]:
!pip install wikipedia==1.4.0
!pip install markitdown
!pip install rich

In [6]:
from getpass import getpass
GENAI_KEY = getpass("enter api")

enter api··········


In [15]:
TAVILY_API_KEY = getpass("enter api")

enter api··········


In [16]:
WEATHER_API_KEY = getpass("enter api")

enter api··········


In [17]:
import os

os.environ['OPENAI_API_KEY'] = GENAI_KEY
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY
os.environ['WEATHER_API_KEY'] = WEATHER_API_KEY

In [18]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_tool = TavilySearchResults(
                                      max_results=8,
                                      search_depth='advanced',
                                      include_raw_content=True
)

In [19]:
tavily_tool.description

'A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. Input should be a search query.'

In [20]:
results = tavily_tool.invoke("tell me about wheather in Manali, Himanchal Pradesh")
results

[{'title': 'Manali, Himachal-Pradesh Weather Forecast | MSN Weather',
  'url': 'https://www.msn.com/en-us/weather/forecast/in-Manali,Himachal-Pradesh',
  'content': '# Manali, Himachal Pradesh, India\n\nCurrent weather\n\n10:58 PM\n\n54°F\n\nCloudy\n\nFeels like\n\n\u200e54°\n\nThunderstorms are expected to move through your area. The low will be 45°.\n\nAir quality\n\nGood air quality (0-50)  \n Primary pollutant O₃ 32 ppb\n\n38\n\nWind\n\nLight air (1 - 3 mph)\n\nDirection shown by smoke drift but not by wind vanes.\n\n2 mph\n\nHumidity\n\nAmount of moisture present in the air relative to the maximum amount of moisture the air can contain at its current temperature.\n\n38%\n\nVisibility\n\nModerate visibility (2 - 5 mi)\n\n2.5 mi\n\nPressure\n\nPressure is the weight of the air in the atmosphere. It is normalized to the standard atmospheric pressure of 1,013.25 mb (29.9212 inHg). Higher pressure is usually associated with sunny weather, lower pressure with stormy weather.\n\n30.26 in

In [21]:
tavily_tool = TavilySearchResults(
                                      max_results=5,
                                      search_depth='advanced',
                                      include_raw_content=True
)

results = tavily_tool.invoke("tell me best place to visit to Himanchanl Pradesh , india ")
results

[{'title': '25 Best Places to Visit in Himachal Pradesh - Tourist Attractions & More',
  'url': 'https://www.thomascook.in/blog/india-holidays/places-to-visit-in-himachal-pradesh',
  'content': '## Best Places to visit in Himachal Pradesh\n\n### 1. Manali\n\nLocated in Kullu district, at an altitude of 2,050 meters, Manali captivates you with its snow-capped mountains and impressive views. Manali is one of the most beautiful places to visit in Himachal. Bring out your adventurous side by indulging in the sports activities as you give yourselves a visual treat. The visually pleasant atmosphere makes it one of  the top tourist places in Himachal Pradesh.\n\n Best Time to Visit:October to February\n Distance from Delhi :559 km\n\n### 2. Khajjiar [...] ### 24. Shoja\n\nUntouched by tourism, Shoja is one of the fine places to visit in Himachal Pradesh, best place to relax and breathe in the fresh mountain air. A perfect escape from the chaotic city life and a very popular Himachal tourist p

In [39]:
import os
import requests
from tqdm import tqdm
from markitdown import MarkItDown
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent

# Retrieve API keys safely from environment variables or global scope
tavily_api_key = os.environ.get("TAVILY_API_KEY", globals().get("TAVILY_API_KEY"))
weather_api_key = os.environ.get("WEATHER_API_KEY", globals().get("WEATHER_API_KEY"))
gemini_api_key = os.environ.get("OPENAI_API_KEY", globals().get("GENAI_KEY"))

# 1. Initialize dependencies & tools
tavily_tool = TavilySearchResults(
    max_results=5,
    search_depth='advanced',
    include_answer=False,
    include_raw_content=True,
    tavily_api_key=tavily_api_key
)
md = MarkItDown()

@tool
def search_web_extract_info(query: str) -> list:
    """Search web and extract detailed information about a given topic or place."""
    results = tavily_tool.invoke(query)
    docs = []
    for result in tqdm(results):
        try:
            extracted_info = md.convert(result['url'])
            text_title = extracted_info.title.strip()
            text_content = extracted_info.text_content.strip()
            docs.append(text_title + '\n' + text_content)
        except:
            print("extraction blocked for url", result["url"])
            pass
    return docs

@tool
def get_weather(query: str) -> dict:
    """Search the weather API to get the current weather for a queried location."""
    print("calling weather tool")
    base_url = "http://api.weatherapi.com/v1/current.json"
    complete_url = f"{base_url}?key={weather_api_key}&q={query}"

    response = requests.get(complete_url)
    data = response.json()
    if data.get("location"):
        return data
    else:
        return {"error": "weather data not found"}

# 2. Initialize the Google Gemini chat model using ChatGoogleGenerativeAI (Using the suggested gemini-3.8-flash)
gemini = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0, google_api_key=gemini_api_key)

# 3. List of tools available for the agent
tools = [search_web_extract_info, get_weather]

# 4. Define system prompt guiding tool selection and observation analysis
system_prompt = (
    "You are a highly capable Travel Assistant AI.\n\n"
    "Your goal is to help users plan trips, extract detailed web destination information, "
    "and check real-time weather conditions using your available tools.\n\n"
    "GUIDELINES FOR TOOL USAGE:\n"
    "1. If the user asks about the current weather, temperature, or climate forecast for a specific city or region, "
    "you MUST call the 'get_weather' tool.\n"
    "2. If the user asks general travel questions, itineraries, places to visit, or background info on attractions, "
    "you MUST call the 'search_web_extract_info' tool.\n"
    "3. Always evaluate the tool's execution result (Observation) carefully. Use the retrieved factual details "
    "to formulate your final response. Do not invent or assume facts outside of the tool output."
)

# 5. Create the tool-calling agent using LangGraph's prebuilt react agent
try:
    agent_executor = create_react_agent(gemini, tools, prompt=system_prompt)
except TypeError:
    agent_executor = create_react_agent(gemini, tools, state_modifier=system_prompt)

/tmp/ipykernel_19771/3545492277.py:77: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(gemini, tools, prompt=system_prompt)


In [40]:
# Ask the agent to check the weather in Manali
query = "What is the current weather like in Manali, Himachal Pradesh?"
print(f"User Query: {query}\n")

# Invoke the compiled agent executor
response = agent_executor.invoke({"messages": [("user", query)]})

# Extract and display the assistant's final response
final_response = response["messages"][-1].content
print("Agent Response:")
print(final_response)

User Query: What is the current weather like in Manali, Himachal Pradesh?

calling weather tool
Agent Response:
[{'type': 'text', 'text': 'The current weather in **Manali, Himachal Pradesh** is:\n\n* **Condition:** Clear\n* **Temperature:** 8.7°C (47.6°F)\n* **Feels Like:** 7.4°C (45.3°F)\n* **Humidity:** 83%\n* **Wind:** 8.3 km/h (5.1 mph) blowing from the northeast (gusts up to 22.7 km/h)\n* **Visibility:** 10 km (6 miles)\n* **Precipitation:** 0 mm (11% chance of rain)', 'extras': {'signature': 'EpUDCpIDAWkUfRMvaNEKbPCmx00HqTthn1+BTTMwfox2WNOsZgDRLcOESQ+y2KgE9jFs9621zds+27CMrPHY0ucaGxHwuWKBdwfEAIfPjyVX7r1cMYJVSPCv8533Dbk+2nodr7asOZnPclKybx0BpFO7TWSw1Xvv6UTQ5GNJY9uh6rcZb0f2FtqI/ywLsXn9CK/QO0YO6qLlLiLUTJ2Z43sx1ys4WW5oEXEPNXYEf6MTpdsUADjOR3rNb8zBkkhSDMq2jREDPyD4yQrUoqCC0e/ZPbsGQuCn/S10wnydPPz5NLUTPLvgYwUCxq9d8C8hAlYbt8lm98uf9jKRoiOlyjj4DOsmJhKgKBlP4RyULOuqVRJ8KDooMFKp4r6SJKgoOg9zqOI4XpbwdVm565Ob/l7rPRakQZo/DFGSbXarGuc81XNiNxGJ2TS/kKxmxwM+p/zau8dRnxLsKEFWumoVsHA3Pssm4VTxx/u6x4xLKAGO4c2Z

In [43]:
from IPython.display import display, Markdown

# Safely display the documents or search results
if 'docs' in globals() and docs:
    print(f"Successfully loaded {len(docs)} extracted documents. Displaying formatting:\n")
    for idx, doc in enumerate(docs, 1):
        display(Markdown(f"## Extracted Document {idx}\n---\n{doc}\n"))
elif 'results' in globals() and results:
    print(f"'docs' was not found, but we have {len(results)} search results available. Displaying formatting:\n")
    for idx, res in enumerate(results, 1):
        title = res.get('title', 'No Title')
        url = res.get('url', '#')
        content = res.get('content', '')
        display(Markdown(f"### {idx}. [{title}]({url})\n\n{content}\n\n---"))
else:
    print("Neither 'docs' nor 'results' variables are found or populated in this session.")

'docs' was not found, but we have 5 search results available. Displaying formatting:



### 1. [25 Best Places to Visit in Himachal Pradesh - Tourist Attractions & More](https://www.thomascook.in/blog/india-holidays/places-to-visit-in-himachal-pradesh)

## Best Places to visit in Himachal Pradesh

### 1. Manali

Located in Kullu district, at an altitude of 2,050 meters, Manali captivates you with its snow-capped mountains and impressive views. Manali is one of the most beautiful places to visit in Himachal. Bring out your adventurous side by indulging in the sports activities as you give yourselves a visual treat. The visually pleasant atmosphere makes it one of  the top tourist places in Himachal Pradesh.

 Best Time to Visit:October to February
 Distance from Delhi :559 km

### 2. Khajjiar [...] ### 24. Shoja

Untouched by tourism, Shoja is one of the fine places to visit in Himachal Pradesh, best place to relax and breathe in the fresh mountain air. A perfect escape from the chaotic city life and a very popular Himachal tourist place. Located in the Seraj Valley, it is famous for its Serloskar Lake and the sunset here is sure to make an unforgettable space in your memory and heart. Shoja adds to the exquisiteness of Himachal tourism.

 Best Time to Visit: May to September
 Distance from Delhi :440 km
 TripAdvisor Rating: 4.5/5

### 25. Barot [...] ### 3. Shimla – Best Places in Himachal

A popular hill station which usually tops the list of places to visit in Himachal Pradesh, Shimla is at an elevation of 2,205 meters. Shimla is the capital and also the largest city of Himachal Pradesh. The lush green environs, scenic landscapes and the wonderful amalgamation of colonial heritage and nature make this place a real treat.  The Viceregal Lodge, Christ Church, Shri Hanuman Jakhoo Temple, Ridge Road and Annadale are some of the popular attractions in Shimla making it one of the most recommended places to visit in Himachal Pradesh.

 Best Time to Visit:March to June
 Distance from Delhi : 346 km
 TripAdvisor Rating: 4.5/5

### 4. Dalhousie

---

### 2. [Places to Visit in Himachal Pradesh - Tourist Places & Things to Do](https://www.makemytrip.com/tripideas/places-to-visit-in-himachal-pradesh)

)

## Best Places to Visit in Himachal Pradesh!

Himachal Pradesh, the northern gem of India, is a haven for nature lovers and adventurers alike. Renowned for its breathtaking landscapes, it features a stunning blend of snow-capped mountains, lush green valleys, mesmerizing rivers, and sprawling forests. This 'Abode of Snow' is home to a collection of multicultural cities like Shimla, Manali, Dharamshala, each bearing its unique charm, history, and experiences. Himachal Pradeshhas something for every traveler. [...] |  |  |
 --- |
|  | Himachal Pradesh |
| Things to Do in Himachal Pradesh | The Ridge-Shimla, Hadimba Temple-Manali, Dalai Lama Temple-Dharamshala, Rohtang Pass-Manali, Solang Valley-Manali |
| Best Time to Visit Himachal Pradesh |  |
| Places to Visit Nearby Himachal Pradesh | Shimla, Manali, Dharamshala, Solan, Kullu |

## Best Places to Visit in Shimla, H.P

The Ridge: A large open space located in theheart of Shimla,known for its amazing views of the mountain ranges.

Mall Road: Shimla's main street filled with shops selling local handicrafts, apparels, and eateries.

Jakhu Temple: A revered hilltop temple dedicated to Lord Hanuman, offering captivating panoramic views of the mountains.

Kufri: A small hill station near Shimla, popular for winter sports like skiing. [...] MMT's LOGO

 Home
 Where2Go
 Places to Visit in Himachal Pradesh

# Places to Visit in Himachal Pradesh

Discover the beauty of Himachal Pradesh with its Manali, Shimla, Dharamshala & Mcleodganj, Kasauli, Kasol. Explore the top places in the state, including Hidimba Devi Temple, Museum of Himachal Culture & Folk Art, Khir Ganga, Nyingmapa Buddhist Temple, Manikaran Sahib, and immerse yourself in a memorable travel

Travelling from

Select City

Manali, Himachal Pradesh)

### 1Manali, Himachal Pradesh

Say hello to hippie villages and scenic valleys.

Nestled on the banks of River Beas, Manali will take your breath away with its snow-kissed peaks, lush valleys and heart-pumping adventure experiences in the heart of Himalayas.

---

### 3. [Best Places To Visit In Himachal Pradesh – FootLoose Dev](https://footloosedev.com/top-tourist-places-himachal-pradesh)

Spiti Valley is (nearly) 100% Buddhist. So, if you want to explore Buddhist culture and some of the oldest monasteries in India, Spiti Valley is the place to be.

Where most people visit Spiti Valley from Manali and return the same way, I advise you to do a complete loop from Shimla to Kaza (Spiti Valley) to Manali. Though the route is a little longer (takes at least 6 days) you get to see and experience most of it.

The best way to explore Spiti is by planning a self-driven road trip. Visit Spiti Valley in summer and visit Spiti Valley in winter to experience two very different sides of it.

### Manali

Next in this list of top popular places to visit in Himachal Pradesh is the town of Manali. And Manali gets a much-deserved place for the variety of experiences it offers. [...] Read my Dharamshala travel blog for more reasons why I think Mcleodgunj, Dharamshala is one of the best places to see in Himachal Pradesh for tourists.

You may also be interested in this detailed Himachal Pradesh travel guide I have written mentioning more details than just top destinations.

## Useful Travel Tips For Himachal Prades

Now that I have discussed some of the popular and offbeat destinations let me share some of the most useful tips to keep in mind. [...] 1. Most places in this Himalayan state don’t accept any form of digital payment.
2. ATMs, outside of the main towns like Shimla, Mandi and Dharamshala can also be rare.
3. The best time to visit Himachal Pradesh is during summer. Winter (unless you want to see snow) can be too harsh to travel around. Monsoon months are also not favorable times to visit as most places become landslide-prone during the rainy season.
4. For a scenic view of snow-covered hills, winter is the best time to visit.
5. Most locals speak Hindi, with very little knowledge of English.
6. Locals here are very welcoming and affable. The entire state is very safe to travel to, even for solo female travellers.

---

### 4. [Himachal Pradesh: Top 10 Best Places to Visit in 2026 | India Travel Guide](https://www.youtube.com/watch?v=dHEVGuw5LCE&vl=en)

# Himachal Pradesh: Top 10 Best Places to Visit in 2026 | India Travel Guide
## Channel: TRIP XTREME (verified)
107K subscribers
128 likes

### Description
20,780 views
Posted: 2024-10-18
Explore Himachal Pradesh: Top 10 Best Places to Visit in 2026 with this exciting India travel guide! From the enchanting peaks of Manali to the serene valleys of Spiti, Himachal Pradesh offers unforgettable experiences for every traveler. Discover historic temples, vibrant hill towns, and breathtaking landscapes that showcase the region’s natural beauty and cultural heritage. Whether you’re an adventurer, a nature lover, or simply seeking tranquility, Himachal has it all. Watch now to plan your perfect escape to India’s stunning hill paradise!
👉 Subscribe to our channel and turn on 🔔

---

### 5. [Himachal Pradesh Tourism - Best Time, Nearby Places & How to Reach](https://www.theindia.co.in/state/himachal-pradesh)

icon19 Places

Dharamshala

### Dharamshala

Are you looking for the best place to relax and unwind, Dharamsala is the perfect place to visit in Himachal Pradesh. Dharamsala is located in Kangra valley in Himachal Pradesh. It is situated at a height of 2090 meters above sea level on a lower slope of the Himalayan mountain. It is also famous for a number of temples and Buddhist culture, which is dominated by the Tibetan people and an abode of the Dalai Llama. People from different parts of India stay in Dharamsala, i.e., Tibetan refugees, Nepali, Gaddi folk, Kashmiri, and other migrant workers.

## Places to Visit in Dharamshala: [...] ## 

## Places to visit in Kullu

### 1. Manikaran

Manikaran is a famous pilgrimage center for Hindus and Sikhs. Manikaran is located at a height of 1,829 mt. also, 40 kms from Kullu. It has the best boiling water springs. Manikaran has a lovely climate throughout the year. Manikaran is Kullu famous place which is surrounded by tourists all-round the year.

### 2. Kheerganga

Kheerganga Trek (Khir Ganga), situated at a height of 2950 m, it is one of the most notable Kullu tourist spots in Himachal Pradesh. The view on the dusk and traveling across Kheerganga backwoods are extraordinary. As said by some old folklore, Kartikeya, the younger child of Lord Shiva and Parvati, contemplated for 1,000 years at Kheerganga making it a vital strict spot for Hindus.

### 3. Manali [...] This city has been a proper summer holiday spot for the past British colonists. but now it has become the number one preference of all Indians to escape the blazing summer.  
Where it is sweaty and exhausting in other parts of the country, people crave to visit Shimla to breathe in some moments of peace.

## Places to Visit in Shimla:

Shimla is considered as the center point from where you can plan your tour towards various region of Himachal pradesh. Some of the best places you can explore in and near to Shimla are:

1. Summer Hills
2. Jakhoo hill
3. Naldhera and Shaily Peak
4. Chadwick falls
5. Johnnie’s Wax Museum
6. Christ Church
7. Himalayan Bird Park
8. Tara Devi Temple

## Top Things to Do in Shimla

---

In [23]:
from langchain_core.tools import tool
from markitdown import MarkItDown
from langchain_community.tools.tavily_search import TavilySearchResults
from tqdm import tqdm
import requests

tavily_tool = TavilySearchResults(
                                      max_results=5,
                                      search_depth='advanced',
                                      include_answer=False,
                                      include_raw_content=True
)
md = MarkItDown()
@tool

def search_web_extract_info(query: str) -> list:
  """ search web and extract information"""
  results = tavily_tool.invoke(query)
  docs = []

  for result in tqdm(results):
    #extract all text column from url

    try:
      extracted_info = md.convert(result['url'])
      text_title = extracted_info.title.strip()
      text_content = extracted_info.text_content.strip()
      docs.append(text_title + '\n' + text_content)
    except:
      print("extraction blocked for url", result["url"])
      pass

  return docs

In [14]:
docs = search_web_extract_info.invoke("tell me best place to visit to Himanchanl Pradesh in india ")

 40%|████      | 2/5 [01:40<02:55, 58.60s/it]

extraction blocked for url https://www.makemytrip.com/tripideas/places-to-visit-in-himachal-pradesh


100%|██████████| 5/5 [01:44<00:00, 20.81s/it]


In [15]:
from IPython.display import display, Markdown
display(Markdown(docs[0]))

25 Best Places to Visit in Himachal Pradesh - Tourist Attractions & More
![](https://blogs.thomascook.in//wp-content/uploads/2024/03/m2i8G6i8G6m2H7b1-removebg-preview.png) 18002099100

[![logo](https://blogs.thomascook.in//wp-content/uploads/2026/01/logo_tc.png)
![dark logo](https://blogs.thomascook.in//wp-content/uploads/2026/01/logo_tc.png) ![light logo](https://blogs.thomascook.in//wp-content/uploads/2026/01/logo_tc.png)](https://www.thomascook.in/blog/)

* [India Holidays](https://www.thomascook.in/blog/india-holidays/)

  + [Top Destinations](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/)
    - [Kerala](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/kerala-top-destination/)
    - [Ladakh](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/ladakh/)
    - [Andaman](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/andaman/)
    - [Goa](https://www.thomascook.in/blog/goa/)
    - [North East](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/north-east-top-destination/)
    - [Rajasthan](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/rajasthan-top-destination/)
    - [Uttrakhand](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/uttrakhand/)
    - [Himachal](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/himachal/)
    - [Kashmir](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/kashmir/)
  + [Beaches](https://www.thomascook.in/blog/india-holidays/for-couples/beaches/)
    - [Maharashtra](https://www.thomascook.in/blog/maharashtra/)
    - [Odisha](https://www.thomascook.in/blog/odisha/)
    - [Pondicherry](https://www.thomascook.in/blog/pondicherry/)
    - [West Bengal](https://www.thomascook.in/blog/west-bengal/)
    - [Andaman](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/andaman/)
    - [Andaman and Lakshadweep](https://www.thomascook.in/blog/india-holidays/andaman-and-lakshadweep/)
    - [Andhra Pradesh](https://www.thomascook.in/blog/andhra-pradesh/)
    - [Gujarat](https://www.thomascook.in/blog/gujarat/)
    - [Karnataka](https://www.thomascook.in/blog/karnataka/)
    - [Kerala](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/kerala-top-destination/)
  + [Hill Station](https://www.thomascook.in/blog/india-holidays/hill-station/)
    - [East](https://www.thomascook.in/blog/india-holidays/hill-station/east/)
    - [West](https://www.thomascook.in/blog/india-holidays/hill-station/west/)
    - [North](https://www.thomascook.in/blog/india-holidays/hill-station/north/)
    - [South](https://www.thomascook.in/blog/india-holidays/hill-station/south/)
  + [Weekend Gateways](https://www.thomascook.in/blog/india-holidays/weekend-gateways/)
    - [Mumbai](https://www.thomascook.in/blog/india-holidays/weekend-gateways/mumbai/)
    - [Chennai](https://www.thomascook.in/blog/india-holidays/weekend-gateways/chennai/)
    - [Delhi](https://www.thomascook.in/blog/india-holidays/weekend-gateways/delhi/)
    - [Bangalore](https://www.thomascook.in/blog/india-holidays/weekend-gateways/bangalore/)
    - [Pune](https://www.thomascook.in/blog/india-holidays/weekend-gateways/pune/)
* [International Holidays](https://www.thomascook.in/blog/international-holidays/)

  + [Top Destinations](https://www.thomascook.in/blog/international-holidays/top-international-destinations/)
    - [Africa](https://www.thomascook.in/blog/international-holidays/africa/)
    - [HongKong](https://www.thomascook.in/blog/international-holidays/hongkong/)
    - [Europe](https://www.thomascook.in/blog/international-holidays/europe/)
    - [Dubai](https://www.thomascook.in/blog/international-holidays/dubai-holidays/)
    - [China](https://www.thomascook.in/blog/international-holidays/china/)
    - [Canada](https://www.thomascook.in/blog/international-holidays/canada-holidays/)
    - [Sri Lanka](https://www.thomascook.in/blog/international-holidays/sri-lanka/)
  + [Top Destinations](https://www.thomascook.in/blog/international-holidays/top-international-destinations/)
    - [Australia](https://www.thomascook.in/blog/international-holidays/australia-holidays/)
    - [USA](https://www.thomascook.in/blog/international-holidays/usa-holidays/)
    - [Thailand](https://www.thomascook.in/blog/international-holidays/thailand/)
    - [Singapore](https://www.thomascook.in/blog/international-holidays/singapore/)
    - [New Zealand](https://www.thomascook.in/blog/international-holidays/new-zealand/)
    - [Mauritius](https://www.thomascook.in/blog/international-holidays/mauritius/)
* Blog by theme

  + Blog By Theme
    - [Adventure](https://www.thomascook.in/blog/adventure/)
    - [Nightlife](https://www.thomascook.in/blog/nightlife/)
    - [Budget Trip](https://www.thomascook.in/blog/budget/)
    - [Festival](https://www.thomascook.in/blog/festival-and-events/)
    - [Food](https://www.thomascook.in/blog/food-blog/)
    - [Shopping](https://www.thomascook.in/blog/offbeat/shopping/)
    - [For Couples](https://www.thomascook.in/blog/honeymoon/)
    - [Scuba Diving](https://www.thomascook.in/blog/scuba-diving/)
  + Blog By Theme
    - [Sky Diving](https://www.thomascook.in/blog/sky-diving/)
    - [Bunjee Jumping](https://www.thomascook.in/blog/bunjee-jumping/)
    - [Hiking](https://www.thomascook.in/blog/hiking/)
    - [Beaches](https://www.thomascook.in/blog/india-holidays/for-couples/beaches/)
    - [Heritage](https://www.thomascook.in/blog/heritage/)
    - [Mountains](https://www.thomascook.in/blog/mountains/)
    - [Trekking](https://www.thomascook.in/blog/trekking/)
* [Forex](https://www.thomascook.in/blog/forex/)

  + [Currency Exchanges](https://www.thomascook.in/blog/forex/currency-exchanges/)
  + [Transfer Money Abroad](https://www.thomascook.in/blog/forex/transfer-money-abroad/)
  + [Forex Card](https://www.thomascook.in/blog/forex/forex-card/)
  + [Study Abroad](https://www.thomascook.in/blog/forex/study-abroad/)

[![Mobile Logo](https://blogs.thomascook.in//wp-content/uploads/2026/01/logo_tc.png)](https://www.thomascook.in/blog/)

* [India Holidays](https://www.thomascook.in/blog/india-holidays/)
  + [Top Destinations](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/)
    - [Kerala](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/kerala-top-destination/)
    - [Ladakh](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/ladakh/)
    - [Andaman](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/andaman/)
    - [Goa](https://www.thomascook.in/blog/goa/)
    - [North East](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/north-east-top-destination/)
    - [Rajasthan](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/rajasthan-top-destination/)
    - [Uttrakhand](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/uttrakhand/)
    - [Himachal](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/himachal/)
    - [Kashmir](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/kashmir/)
  + [Beaches](https://www.thomascook.in/blog/india-holidays/for-couples/beaches/)
    - [Maharashtra](https://www.thomascook.in/blog/maharashtra/)
    - [Odisha](https://www.thomascook.in/blog/odisha/)
    - [Pondicherry](https://www.thomascook.in/blog/pondicherry/)
    - [West Bengal](https://www.thomascook.in/blog/west-bengal/)
    - [Andaman](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/andaman/)
    - [Andaman and Lakshadweep](https://www.thomascook.in/blog/india-holidays/andaman-and-lakshadweep/)
    - [Andhra Pradesh](https://www.thomascook.in/blog/andhra-pradesh/)
    - [Gujarat](https://www.thomascook.in/blog/gujarat/)
    - [Karnataka](https://www.thomascook.in/blog/karnataka/)
    - [Kerala](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/kerala-top-destination/)
  + [Hill Station](https://www.thomascook.in/blog/india-holidays/hill-station/)
    - [East](https://www.thomascook.in/blog/india-holidays/hill-station/east/)
    - [West](https://www.thomascook.in/blog/india-holidays/hill-station/west/)
    - [North](https://www.thomascook.in/blog/india-holidays/hill-station/north/)
    - [South](https://www.thomascook.in/blog/india-holidays/hill-station/south/)
  + [Weekend Gateways](https://www.thomascook.in/blog/india-holidays/weekend-gateways/)
    - [Mumbai](https://www.thomascook.in/blog/india-holidays/weekend-gateways/mumbai/)
    - [Chennai](https://www.thomascook.in/blog/india-holidays/weekend-gateways/chennai/)
    - [Delhi](https://www.thomascook.in/blog/india-holidays/weekend-gateways/delhi/)
    - [Bangalore](https://www.thomascook.in/blog/india-holidays/weekend-gateways/bangalore/)
    - [Pune](https://www.thomascook.in/blog/india-holidays/weekend-gateways/pune/)
* [International Holidays](https://www.thomascook.in/blog/international-holidays/)
  + [Top Destinations](https://www.thomascook.in/blog/international-holidays/top-international-destinations/)
    - [Africa](https://www.thomascook.in/blog/international-holidays/africa/)
    - [HongKong](https://www.thomascook.in/blog/international-holidays/hongkong/)
    - [Europe](https://www.thomascook.in/blog/international-holidays/europe/)
    - [Dubai](https://www.thomascook.in/blog/international-holidays/dubai-holidays/)
    - [China](https://www.thomascook.in/blog/international-holidays/china/)
    - [Canada](https://www.thomascook.in/blog/international-holidays/canada-holidays/)
    - [Sri Lanka](https://www.thomascook.in/blog/international-holidays/sri-lanka/)
  + [Top Destinations](https://www.thomascook.in/blog/international-holidays/top-international-destinations/)
    - [Australia](https://www.thomascook.in/blog/international-holidays/australia-holidays/)
    - [USA](https://www.thomascook.in/blog/international-holidays/usa-holidays/)
    - [Thailand](https://www.thomascook.in/blog/international-holidays/thailand/)
    - [Singapore](https://www.thomascook.in/blog/international-holidays/singapore/)
    - [New Zealand](https://www.thomascook.in/blog/international-holidays/new-zealand/)
    - [Mauritius](https://www.thomascook.in/blog/international-holidays/mauritius/)
* ###### Blog by theme

  + Blog By Theme
    - [Adventure](https://www.thomascook.in/blog/adventure/)
    - [Nightlife](https://www.thomascook.in/blog/nightlife/)
    - [Budget Trip](https://www.thomascook.in/blog/budget/)
    - [Festival](https://www.thomascook.in/blog/festival-and-events/)
    - [Food](https://www.thomascook.in/blog/food-blog/)
    - [Shopping](https://www.thomascook.in/blog/offbeat/shopping/)
    - [For Couples](https://www.thomascook.in/blog/honeymoon/)
    - [Scuba Diving](https://www.thomascook.in/blog/scuba-diving/)
  + ###### Blog By Theme

    - [Sky Diving](https://www.thomascook.in/blog/sky-diving/)
    - [Bunjee Jumping](https://www.thomascook.in/blog/bunjee-jumping/)
    - [Hiking](https://www.thomascook.in/blog/hiking/)
    - [Beaches](https://www.thomascook.in/blog/india-holidays/for-couples/beaches/)
    - [Heritage](https://www.thomascook.in/blog/heritage/)
    - [Mountains](https://www.thomascook.in/blog/mountains/)
    - [Trekking](https://www.thomascook.in/blog/trekking/)
* [Forex](https://www.thomascook.in/blog/forex/)
  + [Currency Exchanges](https://www.thomascook.in/blog/forex/currency-exchanges/)
  + [Transfer Money Abroad](https://www.thomascook.in/blog/forex/transfer-money-abroad/)
  + [Forex Card](https://www.thomascook.in/blog/forex/forex-card/)
  + [Study Abroad](https://www.thomascook.in/blog/forex/study-abroad/)

Top

![](https://www.thomascook.in/blog/wp-content/uploads/2017/07/01-manali.webp)

[Home](https://www.thomascook.in/)
 >
[Blog](https://www.thomascook.in/blog/)
 >
[Top Domestic Destinations](https://www.thomascook.in/blog/india-holidays/top-domestic-destinations/)
 >
25 Best Places to Visit in Himachal Pradesh

# 25 Best Places to Visit in Himachal Pradesh

February 5, 2026 -   9 min read

### Table of contents

1

0

Reading Time:   9 minutes

If you are someone who loves cold climate, tranquillity, a little adventure and a whole lot of fun, then a visit to the hill stations in North India is a must. Make a list of places to visit in Himachal Pradesh where overwhelming landscapes and peaceful views greet you. This place is surely a nature lover’s delight! There is no better way to explore the majestic mountains, evergreen forests, remote villages and mesmerising lakes at its best than taking a trip to “The Land of Snow” which is what Himachal means.

## List of Places to Visit in Himachal Pradesh

|  |  |
| --- | --- |
| **Places to Visit in Himachal Pradesh** | **Distance from Delhi** |
| Manali | 559 km |
| Khajjiar | 575 km |
| Shimla | 346 km |
| Dalhousie | 562 km |
| Parwanoo | 268 km |
| Sirmaur | 293 km |
| Kasauli | 291 km |
| Kasol | 520 km |
| Tirthan Valley | 494 km |
| Sangla | 567 km |
| Una | 353 km |
| Kaza | 742 km |
| Nako Lake | 675 km |
| Chandra Taal | 750 km |
| Lama Dal | 596.7 km |
| Malana | 528 km |
| Spiti | 753 km |
| Dharamsala | 474 km |
| Chail | 339 km |
| Kalpa | 585 km |
| Solan | 585 km |
| Hamirpur | 299 km |
| Thanedar | 420 kmq |
| Shoja | 440 km |
| Barot | 505 km |

## Best Places to visit in Himachal Pradesh

### 1. Manali

![](https://blogs.thomascook.in//wp-content/uploads/2020/10/01-manali.jpg)

Located in Kullu district, at an altitude of 2,050 meters, Manali captivates you with its snow-capped mountains and impressive views. Manali is one of the most beautiful places to visit in Himachal. Bring out your adventurous side by indulging in the sports activities as you give yourselves a visual treat. The visually pleasant atmosphere makes it one of  the top tourist places in Himachal Pradesh.

* **Best Time to Visit:**October to February
* **Distance from Delhi** **:**559 km

### 2. Khajjiar

Khajjiar is a hill station located in the northern state of Himachal Pradesh. At an altitude of 2,000 meters, Khajjiar is situated in the Chamba District. With lakes, pastures, forests and an opportunity for camping in the wild, you will never get bored if your **Himachal tour packages** include a visit to this place. The adventure activities, mesmerizing nature, wildlife and culture makes Khajjiar a must visit place in Himachal.

* **Best Time to Visit:** October to March
* **Distance from Delhi :** 575 km
* **TripAdvisor Rating:** 4/5

### 3. Shimla – Best Places in Himachal

![](https://blogs.thomascook.in//wp-content/uploads/2020/10/02-shimla-1024x768.jpg)

A popular hill station which usually tops the list of **places to visit in Himachal Pradesh**, [Shimla](https://blogs.thomascook.in//20-best-things-to-do-in-shimla) is at an elevation of 2,205 meters. Shimla is the capital and also the largest city of Himachal Pradesh. The lush green environs, scenic landscapes and the wonderful amalgamation of colonial heritage and nature make this place a real treat.  The Viceregal Lodge, Christ Church, Shri Hanuman Jakhoo Temple, Ridge Road and Annadale are some of the popular attractions in Shimla making it one of the most recommended places to visit in Himachal Pradesh.

* **Best Time to Visit:**March to June
* **Distance from Delhi :** 346 km
* **TripAdvisor Rating:** 4.5/5

### 4. Dalhousie

True to its name, Dalhousie exudes a colonial feeling and is renowned for its heritage buildings that go way back to the early 19th century. The colonial feel is good for a walk down memory lane and hence is one of the best tourist places in Himachal Pradesh.  This hill station is spread across five hills and is a sanctuary for wildlife lovers and landscape photographers as it holds some of the most intriguing flora and fauna.

* **Best Time to Visit:** March to May
* **Distance from Delhi:** 562 km
* **TripAdvisor Rating:** 4/5

### 5. Parwanoo

Set against the backdrop of breathtaking hills, Parwanoo lies to the border of Haryana and is home to lush greenery. It is famous for its fruit orchards, timber trails, cable car rides and is also very close to Shimla & Kasauli. This makes it one of the places to visit in Himachal for trekkers.

* **Best Time to Visit:** March to June
* **Distance from Delhi :**268 km
* **TripAdvisor Rating:**4.5/5

### 6. Sirmaur

With fossils that date way back to 85 million years, Sirmaur is located more to the south eastern part of the state. It has great heritage value, beautiful temples and is even known for its shopping.It is largely mountainous and rural, with 90% of its population living in villages. Sirmaur is known for its heritage value, beautiful temples and is even known for its shopping. Sirmaur is one of the most popular Himachal tourist places.

* **Best Time to Visit:** Throughout the year
* **Distance from Delhi :**293 km
* **TripAdvisor Rating:** 4/5

### 7. Kasauli

Set at an altitude of 1,927 meters, Kasauli too reeks of a typical **Himachal temperature**. It is a colonial landmark that is popular for its Central Research Institute, Baptist Church, Kasauli Brewery, Monkey Point and Kasauli Club. This small town in the southwestern part of the state is settled among delightful pine woods and cedar trees. This tourist place in Himachal Pradesh is known for its mystical and serene ambience.

* **Best Time to Visit:** April to June , September to November
* **Distance from Delhi :**291 km
* **TripAdvisor Rating:** 4/5

### 8. Kasol

![](https://blogs.thomascook.in//wp-content/uploads/2020/10/03-kasol.jpg)

Located in the Parvati Valley at an altitude of 1,650 meters, Kasol looks like a perfect picture come to life. It is slowly gaining massive popularity as one of the most **popular tourist places in Himachal Pradesh** and is a base camp for moderate treks in the state. This Himachal tourist places is best known for its thrilling treks, flavoursome Israeli food, alternative hippie lifestyle and culture and the streaming Parvati river. This makes Kasol one of the top contenders for must visit tourist places in Himachal.

* **Best Time to Visit:** October to June
* **Distance from Delhi:** 520 km
* **TripAdvisor Rating:**4.5/5

### 9. **Tirthan Valley**

![](https://blogs.thomascook.in//wp-content/uploads/2020/10/04-tirthan-valley.jpg)

If you want to experience surreal nature along with bone-chilling water cascading down rocky brooks, Tirthan Valley is one of popular places to visit in Himachal Pradesh. The valley is immensely beautiful, it gives you a chance to witness a beautiful rainbow or brown trout in the glacial waters. Trekking, overnight camping, river crossing, night treks, mountain biking, kayaking, and archery are the best ways to relish the picturesque magnificence of the Tirthan Valley.

* **Best Time to Visit:** October to March
* **Distance from Delhi :** 494 km
* **TripAdvisor Rating:** 5/5

### 10. Sangla

With scenic attractions, nature walks and trekking options that enthral you no end, Sangla is known for its apple orchards, cattle farms and beautiful sceneries. If you want to experience calm, rustic life, you need to head here.

* **Best Time to Visit:** March to June, September to December
* **Distance from Delhi :**567 km
* **TripAdvisor Rating:** 4.5/5

### 11. Una

Una is the hottest district in this state and is known as the Gateway to the Himalayas. At 1210 feet, the place sees travellers checking out the Chintpurni Temple. Pong Dam, Maa Chintpurni Temple, Thaneek Pura and Kila Baba Bedi Ji are some of the popular tourist places in Una, Himachal Pradesh.However, if you are planning to visit the place during the summer, be sure to carry cotton clothes with you.

* **Best Time to Visit:** March to May
* **Distance from Delhi :** 353 km
* **TripAdvisor Rating:**4.5/5

### 12. Kaza

![](https://blogs.thomascook.in//wp-content/uploads/2020/10/05-kaza.jpg)

One of the remote villages of Himachal Pradesh, Kaza is set in the Lahaul-Spiti District. Over here resides the Sakya Tangyud Monastery and a few other major attractions.

* **Best Time to Visit:** May to October
* **Distance from Delhi :** 742 km
* **TripAdvisor Rating:**4.5/5

### 13. Nako Lake

Among the **places to visit in Himachal Pradesh**, if you want to travel unexplored paths, the Nako Lake remains secluded from the general public. Enveloped by poplar and willow trees, this small lake is as striking as it is enchanting. There are also four Buddhist temples and the Nako Monastery nearby.

* **Best Time to Visit:** April to October
* **Distance from Delhi Airport:** 675 km
* **TripAdvisor Rating:** 4/5

### 14. Chandra Taal

A barren landscape amalgamated together with a sweet water lake at an altitude of 4,300 meters has never looked more appealing. The crescent shaped lake on the Samudra Tapu Plateau or Moon Lake or the Chandra Taal is a hub for trekkers and campers everywhere. One must definitely take a dip in the lake at this tourist place in Himachal Pradesh.

* **Best Time to Visit:** September
* **Distance from Delhi :** 750 km
* **TripAdvisor Rating:** 5/5

### 15. Lama Dal

Situated near to the Balaini Pass at an altitude of 3,990 meters, the Lama Dal is said to be a sacred lake. It is one of the seven tarns, and you can even visit a temple dedicated to Lord Shiva here.

* **Best Time to Visit:** May to November
* **Distance from Delhi :** 596.7 km

### 16. Malana

If you are looking to get away from the world and its digital attachments, Malana is your safe refuge that is located near the Kullu Valley. You can also take advantage of the Chandrakhani crests and Deo Tibba that are highlights of the place.

* **Best Time to Visit:** May to August
* **Distance from Delhi :** 528 km
* **TripAdvisor Rating:** 5/5

### 17. Spiti

![](https://blogs.thomascook.in//wp-content/uploads/2020/10/06-spiti.jpg)

One of the hidden beauties of North India is the Spiti Valley. Spiti Valley is a snow cladded desert mountain located high in the Himalayas in the north-eastern state of Himachal Pradesh. The name “Spiti” signifies as “The middle land” as the valley is situated between India and Tibet. Spiti Valley is one of the well-known tourist places in Himachal pradesh. This is where divine Buddhist destinations and a whole lot of traveller attractions are unveiled.

* **Best Time to Visit:** May to October
* **Distance from Delhi Airport:**753 km

### 18. Dharamsala

**Himachal Pradesh Tourism** always stresses on visiting Dharamsala as it is here that you get to witness an amazing climate and multicultural Indian communities peacefully live together with the Indo Tibetan community. Dharamshala is also home to the Dalai Lama and is one of the most frequented tourist places to visit in Himachal and known for its particular mix of the Kangra and Tibetan culture.

* **Best Time to Visit:** March to June; December to February
* **Distance from Delhi :** 474 km
* **TripAdvisor Rating:**

### 19. Chail

Chail is one of the famous **hill stations in Himachal** Pradesh. Home to the highest cricket ground, it regularly witnesses cricket enthusiasts from all over the world visit this place. The charming views of Kasauli, Shimla and Solan and the lovely atmosphere of the place are a treat to the senses.

* **Best Time to Visit:** March to May
* **Distance from Delhi :** 339 km
* **TripAdvisor Rating:** 4.5 /5

### 20. Kalpa

One of the biggest villages in the Kinnaur District, Kalpa is known for its plush green forests. Another thing that will take your breath away is the snow covered landscape that Kinnaur Kailash paints when bathed in the morning sunshine. The endless apple orchards add to the marvellous colour palette that Kalpa offers. Kalpa is a pride of [Himachal tourism](https://www.thomascook.in/india-tourism/himachal-tourism) and you definitely must not miss it.

* **Best Time to Visit:** June to Septmeber
* **Distance from Delhi :** 585 km
* **TripAdvisor Rating:** 4.5/5

### 21. Solan

If you have made a list of **places to visit in Himachal Pradesh**, you must definitely not miss this one! Also known as the ‘Mushroom City of India’ and ‘City of Red Gold’, Solan is heaven on earth. You will believe this yourself, once you check out the snow-clad mountains, forests and mushroom and tomato production.

* **Best Time to Visit:** October to May
* **Distance from Delhi :** 585 km
* **TripAdvisor Rating:** 4.5/5

### 22. Hamirpur

If you like to trek and are game for some adventure, you must visit Hamirpur. It is warmer compared to other places in the region as it comes at a lower elevation. Altitudes vary as you move from flat lands to higher levels that touch the heights of cliffs.

There is never a lack of activities to carry out or places to visit in Himachal Pradesh. Find the places that best suit your preferences, likes and your kind of adventure and you will surely have the time of your life.

* **Best Time to Visit:** February to August, September to November
* **Distance from Delhi :** 299 km
* **TripAdvisor Rating:** 4/5

### 23. Thanedar

Alive and soul touching with its apple and cherry orchards, Thanedar is an unparalleled joy to nature lovers. Thanedar is a host to many resorts overlooking the plush orchards that give the perfect experience of staying close to nature. You cannot step out of Thanedar without visiting St. Mary’s church which also happens to be one of the oldest churches in India and Hatu Peak that provides a mesmerising vista of the snow-clad peaks of the Himalayas. Thanedar is one of the best [places to visit in Himachal Pradesh.](https://www.thomascook.in/india-tourism/himachal-tourism/places-to-visit-in-himachal)

* **Best Time to Visit:** October to March, except January
* **Distance from Delhi :** 420 km
* **TripAdvisor Rating:**4/5

### 24. Shoja

Untouched by tourism, Shoja is one of the fine places to visit in Himachal Pradesh, best place to relax and breathe in the fresh mountain air. A perfect escape from the chaotic city life and a very popular Himachal tourist place. Located in the Seraj Valley, it is famous for its Serloskar Lake and the sunset here is sure to make an unforgettable space in your memory and heart. Shoja adds to the exquisiteness of Himachal tourism.

* **Best Time to Visit:** May to September
* **Distance from Delhi :**440 km
* **TripAdvisor Rating:** 4.5/5

### 25. **Barot**

Barot is the epitome of positivity and beautiful vibes. Across from the Uhl River is located Nargu wildlife sanctuary, which is home to a variety of unique Himalayan beasts. The valley is full of evergreen forests with umpteen trekking trails that cut through the sanctuary to Kullu. The temple of the local deity of rain, Dev Pashakot that makes for a fine place to visit, is situated across the river Uhl. Barot is one of the most mesmerising off-beat places to visit in Himachal Pradesh.

* **Best Time to Visit:** April to June, November to February
* **Distance from Delhi :** 505 km
* **TripAdvisor Rating:**5/5

[![Andaman and Nicobar Islands ](https://www.thomascook.in/blog/wp-content/uploads/2025/03/9-4.webp)

##### previous post](https://www.thomascook.in/blog/india-holidays/15-best-places-to-visit-in-april/)

[![Kashmir ](https://www.thomascook.in/blog/wp-content/uploads/2025/03/9-2.webp)

##### next post](https://www.thomascook.in/blog/india-holidays/top-10-places-to-visit-in-june/)

### Table of contents

### Popular India Destinations

* [Manali Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/manali-tour-packages)
* [Andaman Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/andaman-and-nicobar-tour-packages)
* [Kerala Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/kerala-tour-packages)
* [Rajasthan Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/rajasthan-tour-packages)
* [Jammu & Kashmir Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/jammu-and-kashmir-tour-packages)
* [Ladakh Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/ladakh-tour-packages)
* [Himachal Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/himachal-pradesh-tour-packages)
* [Bhutan Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/bhutan-tour-packages)
* [Goa Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/goa-tour-packages)
* [North East Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/north-east-tour-packages)
* [Lakshadweep Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/lakshadweep-tour-packages)

[View All](https://www.thomascook.in/holidays/india-tour-packages)

![](https://blogs.thomascook.in//wp-content/uploads/2024/01/mail-icon-new.png)

### Receive copy of our mailing list

Sign up and receive our monthly insight, travel inspo, plans directly into your inbox.

 Your email\*
  Full Name\*

* [International Holidays](https://www.thomascook.in/blog/international-holidays/)

## [Chasing the Lights Across the World’s Best Northern Lights Countries](https://www.thomascook.in/blog/international-holidays/worlds-best-northern-light-countries/)

By thomascook seo

[![Chasing the Lights Across the World’s Best Northern Lights Countries](https://www.thomascook.in/blog/wp-content/uploads/2026/10/aurora-borealis.webp)](https://www.thomascook.in/blog/international-holidays/worlds-best-northern-light-countries/)

* [International Events](https://www.thomascook.in/blog/international-events/)
* [International Holidays](https://www.thomascook.in/blog/international-holidays/)
* [Japan](https://www.thomascook.in/blog/international-holidays/japan/)

## [Cherry Blossom Festival in Japan – The Ultimate Spring Escape](https://www.thomascook.in/blog/international-holidays/discover-the-cherry-blossom-festival-in-japan-during-the-cherry-blossoms-festival-season-learn-about-sakura-celebrations-viewing-spots-and-best-time-to-visit/)

By Thomas Cook Admin

[![Cherry Blossom Festival in Japan – The Ultimate Spring Escape](https://www.thomascook.in/blog/wp-content/uploads/2018/01/Cherry-Blossom-Festival-In-Japan-1.webp)](https://www.thomascook.in/blog/international-holidays/discover-the-cherry-blossom-festival-in-japan-during-the-cherry-blossoms-festival-season-learn-about-sakura-celebrations-viewing-spots-and-best-time-to-visit/)

* [International Events](https://www.thomascook.in/blog/international-events/)

## [Sakura Cherry Blossom – Witness Japan’s Pink Spring](https://www.thomascook.in/blog/international-holidays/the-spell-of-sakura-cherry-blossoms-in-japan/)

By Thomas Cook Admin

[![Sakura Cherry Blossom – Witness Japan’s Pink Spring](https://www.thomascook.in/blog/wp-content/uploads/2019/12/Cherry-Blossom-Festival3.webp)](https://www.thomascook.in/blog/international-holidays/the-spell-of-sakura-cherry-blossoms-in-japan/)

* [India Holidays](https://www.thomascook.in/blog/india-holidays/)

## [Places to Visit in India in December](https://www.thomascook.in/blog/india-holidays/places-to-visit-in-india-in-december/)

By Thomas Cook Admin

[![Places to Visit in India in December](https://www.thomascook.in/blog/wp-content/uploads/2019/12/Andaman-Nicobar-Tour-Packages-e1504182713890.webp)](https://www.thomascook.in/blog/india-holidays/places-to-visit-in-india-in-december/)

* [India Holidays](https://www.thomascook.in/blog/india-holidays/)

## [12 Best Places To Visit In November In India In 2026](https://www.thomascook.in/blog/india-holidays/12-best-places-to-visit-in-november-2026/)

By Thomas Cook Admin

[![12 Best Places To Visit In November In India In 2026](https://www.thomascook.in/blog/wp-content/uploads/2025/03/1-10.webp)](https://www.thomascook.in/blog/india-holidays/12-best-places-to-visit-in-november-2026/)

* [Forex](https://www.thomascook.in/blog/forex/)

## [What is the average cost of living in Singapore in 2026?](https://www.thomascook.in/blog/forex/what-is-the-average-cost-of-living-in-singapore-in-2026/)

By Thomas Cook Admin

* [Forex](https://www.thomascook.in/blog/forex/)

## [Top Universities in Germany for International Students](https://www.thomascook.in/blog/forex/top-universities-in-germany-for-international-students/)

By Thomas Cook Admin

* [Forex](https://www.thomascook.in/blog/forex/)

## [RBI Forex Guidelines for Foreign Exchange Transactions in India](https://www.thomascook.in/blog/forex/rbi-forex-guidelines-for-foreign-exchange-transactions-in-india/)

By Thomas Cook Admin

* [Forex](https://www.thomascook.in/blog/forex/)

## [Cost of Living Index by Country: Compare Living Expenses Around the World (2026 Guide)](https://www.thomascook.in/blog/forex/cost-of-living-index-by-country-compare-living-expenses-around-the-world-2026-guide/)

By Thomas Cook Admin

* [India Holidays](https://www.thomascook.in/blog/india-holidays/)

## [Delhi To Haridwar distance: Best Route, Train Timings & Travel Guide](https://www.thomascook.in/blog/india-holidays/delhi-to-haridwar-distance-best-route-train-timings-travel-guide/)

By Thomas Cook Admin

[![](https://blogs.thomascook.in//wp-content/uploads/2024/04/Exotic-Kashmir.png)](https://www.thomascook.in/holidays/india-tour-packages/jammu-and-kashmir-tour-packages)

![](https://blogs.thomascook.in//wp-content/uploads/2024/03/m2i8G6i8G6m2H7b1-removebg-preview.png) Call Us

Let's Plan Your trip

### Popular International Destinations

* [Europe Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/europe-tour-packages)
* [Singapore Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/singapore-tour-packages)
* [Thailand Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/thailand-tour-packages)
* [Bali Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/bali-tour-packages)
* [Dubai Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/dubai-tour-packages)
* [Egypt Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/egypt-tour-packages)
* [Turkey Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/turkey-tour-packages)
* [Japan Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/japan-tour-packages)
* [Australia Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/australia-tour-packages)
* [Mauritius Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/mauritius-tour-packages)
* [Vietnam Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/vietnam-tour-packages)

[View All](https://www.thomascook.in/holidays/international-tour-packages)

![](https://blogs.thomascook.in//wp-content/uploads/2024/01/mail-icon-new.png)

### Receive copy of our mailing list

Sign up and receive our monthly insight, travel inspo, plans directly into your inbox.

 Your email\*
  Full Name\*

* [International Holidays](https://www.thomascook.in/blog/international-holidays/)

## [Chasing the Lights Across the World’s Best Northern Lights Countries](https://www.thomascook.in/blog/international-holidays/worlds-best-northern-light-countries/)

By thomascook seo

[![Chasing the Lights Across the World’s Best Northern Lights Countries](https://www.thomascook.in/blog/wp-content/uploads/2026/10/aurora-borealis.webp)](https://www.thomascook.in/blog/international-holidays/worlds-best-northern-light-countries/)

* [International Events](https://www.thomascook.in/blog/international-events/)
* [International Holidays](https://www.thomascook.in/blog/international-holidays/)
* [Japan](https://www.thomascook.in/blog/international-holidays/japan/)

## [Cherry Blossom Festival in Japan – The Ultimate Spring Escape](https://www.thomascook.in/blog/international-holidays/discover-the-cherry-blossom-festival-in-japan-during-the-cherry-blossoms-festival-season-learn-about-sakura-celebrations-viewing-spots-and-best-time-to-visit/)

By Thomas Cook Admin

[![Cherry Blossom Festival in Japan – The Ultimate Spring Escape](https://www.thomascook.in/blog/wp-content/uploads/2018/01/Cherry-Blossom-Festival-In-Japan-1.webp)](https://www.thomascook.in/blog/international-holidays/discover-the-cherry-blossom-festival-in-japan-during-the-cherry-blossoms-festival-season-learn-about-sakura-celebrations-viewing-spots-and-best-time-to-visit/)

* [International Events](https://www.thomascook.in/blog/international-events/)

## [Sakura Cherry Blossom – Witness Japan’s Pink Spring](https://www.thomascook.in/blog/international-holidays/the-spell-of-sakura-cherry-blossoms-in-japan/)

By Thomas Cook Admin

[![Sakura Cherry Blossom – Witness Japan’s Pink Spring](https://www.thomascook.in/blog/wp-content/uploads/2019/12/Cherry-Blossom-Festival3.webp)](https://www.thomascook.in/blog/international-holidays/the-spell-of-sakura-cherry-blossoms-in-japan/)

* [India Holidays](https://www.thomascook.in/blog/india-holidays/)

## [Places to Visit in India in December](https://www.thomascook.in/blog/india-holidays/places-to-visit-in-india-in-december/)

By Thomas Cook Admin

[![Places to Visit in India in December](https://www.thomascook.in/blog/wp-content/uploads/2019/12/Andaman-Nicobar-Tour-Packages-e1504182713890.webp)](https://www.thomascook.in/blog/india-holidays/places-to-visit-in-india-in-december/)

* [India Holidays](https://www.thomascook.in/blog/india-holidays/)

## [12 Best Places To Visit In November In India In 2026](https://www.thomascook.in/blog/india-holidays/12-best-places-to-visit-in-november-2026/)

By Thomas Cook Admin

[![12 Best Places To Visit In November In India In 2026](https://www.thomascook.in/blog/wp-content/uploads/2025/03/1-10.webp)](https://www.thomascook.in/blog/india-holidays/12-best-places-to-visit-in-november-2026/)

* [Forex](https://www.thomascook.in/blog/forex/)

## [What is the average cost of living in Singapore in 2026?](https://www.thomascook.in/blog/forex/what-is-the-average-cost-of-living-in-singapore-in-2026/)

By Thomas Cook Admin

* [Forex](https://www.thomascook.in/blog/forex/)

## [Top Universities in Germany for International Students](https://www.thomascook.in/blog/forex/top-universities-in-germany-for-international-students/)

By Thomas Cook Admin

* [Forex](https://www.thomascook.in/blog/forex/)

## [RBI Forex Guidelines for Foreign Exchange Transactions in India](https://www.thomascook.in/blog/forex/rbi-forex-guidelines-for-foreign-exchange-transactions-in-india/)

By Thomas Cook Admin

* [Forex](https://www.thomascook.in/blog/forex/)

## [Cost of Living Index by Country: Compare Living Expenses Around the World (2026 Guide)](https://www.thomascook.in/blog/forex/cost-of-living-index-by-country-compare-living-expenses-around-the-world-2026-guide/)

By Thomas Cook Admin

* [India Holidays](https://www.thomascook.in/blog/india-holidays/)

## [Delhi To Haridwar distance: Best Route, Train Timings & Travel Guide](https://www.thomascook.in/blog/india-holidays/delhi-to-haridwar-distance-best-route-train-timings-travel-guide/)

By Thomas Cook Admin

[![](https://blogs.thomascook.in/wp-content/uploads/2026/07/vietnam.jpg)](https://www.thomascook.in/holidays/international-tour-packages/vietnam-tour-packages?utm_source=google&utm_medium=blog&utm_campaign=promotion)

![](https://blogs.thomascook.in//wp-content/uploads/2024/03/m2i8G6i8G6m2H7b1-removebg-preview.png) Call Us

Let's Plan Your trip

### Recommended Articles for you

[![](https://www.thomascook.in/blog/wp-content/uploads/2026/10/aurora-borealis.webp)](https://www.thomascook.in/blog/international-holidays/worlds-best-northern-light-countries/)

## [Chasing the Lights Across the World’s Best Northern Lights Countries](https://www.thomascook.in/blog/international-holidays/worlds-best-northern-light-countries/)

* [thomascook seo](https://www.thomascook.in/blog/author/seo/)
* October 7, 2026

0 0 There is a particular kind of silence that falls over you the first time the sky turns green above your head. It is not something a screen can prepare you for. That unforgettable moment is what makes Northern […]

[Read More](https://www.thomascook.in/blog/international-holidays/worlds-best-northern-light-countries/)

[![Cherry Blossom Festival In Japan](https://www.thomascook.in/blog/wp-content/uploads/2018/01/Cherry-Blossom-Festival-In-Japan-1.webp)](https://www.thomascook.in/blog/international-holidays/discover-the-cherry-blossom-festival-in-japan-during-the-cherry-blossoms-festival-season-learn-about-sakura-celebrations-viewing-spots-and-best-time-to-visit/)

## [Cherry Blossom Festival in Japan – The Ultimate Spring Escape](https://www.thomascook.in/blog/international-holidays/discover-the-cherry-blossom-festival-in-japan-during-the-cherry-blossoms-festival-season-learn-about-sakura-celebrations-viewing-spots-and-best-time-to-visit/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* October 7, 2026
* [3](https://www.thomascook.in/blog/international-holidays/discover-the-cherry-blossom-festival-in-japan-during-the-cherry-blossoms-festival-season-learn-about-sakura-celebrations-viewing-spots-and-best-time-to-visit/)

0 0 If there is one country that can consistently surprise you, enchant you and make you feel blessed about life, it has to be Japan. From retaining deep-rooted cultural beliefs to staying ahead in technology and being a pioneer […]

[Read More](https://www.thomascook.in/blog/international-holidays/discover-the-cherry-blossom-festival-in-japan-during-the-cherry-blossoms-festival-season-learn-about-sakura-celebrations-viewing-spots-and-best-time-to-visit/)

[![](https://www.thomascook.in/blog/wp-content/uploads/2019/12/Cherry-Blossom-Festival3.webp)](https://www.thomascook.in/blog/international-holidays/the-spell-of-sakura-cherry-blossoms-in-japan/)

## [Sakura Cherry Blossom – Witness Japan’s Pink Spring](https://www.thomascook.in/blog/international-holidays/the-spell-of-sakura-cherry-blossoms-in-japan/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* October 7, 2026
* [0](https://www.thomascook.in/blog/international-holidays/the-spell-of-sakura-cherry-blossoms-in-japan/)

0 0 Every year, Japan’s gardens, trees and parks turn from a glorious green of jade to the soft pink of a blushing bride. That’s right, the cherry blossoms are ready to bloom, bringing with it the promise of a […]

[Read More](https://www.thomascook.in/blog/international-holidays/the-spell-of-sakura-cherry-blossoms-in-japan/)

[![](https://www.thomascook.in/blog/wp-content/uploads/2019/12/Andaman-Nicobar-Tour-Packages-e1504182713890.webp)](https://www.thomascook.in/blog/india-holidays/places-to-visit-in-india-in-december/)

## [Places to Visit in India in December](https://www.thomascook.in/blog/india-holidays/places-to-visit-in-india-in-december/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* October 6, 2026
* [0](https://www.thomascook.in/blog/india-holidays/places-to-visit-in-india-in-december/)

0 0 December marks the beginning of the peak travel season in India. From the North of the country to the South, every region has pleasant weather and clear skies, perfect for sightseeing. After the hot summer and muggy monsoon, […]

[Read More](https://www.thomascook.in/blog/india-holidays/places-to-visit-in-india-in-december/)

[![Featured img](https://www.thomascook.in/blog/wp-content/uploads/2025/03/1-10.webp)](https://www.thomascook.in/blog/india-holidays/12-best-places-to-visit-in-november-2026/)

## [12 Best Places To Visit In November In India In 2026](https://www.thomascook.in/blog/india-holidays/12-best-places-to-visit-in-november-2026/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* October 6, 2026
* [0](https://www.thomascook.in/blog/india-holidays/12-best-places-to-visit-in-november-2026/)

0 0 November is a month of vibrant transitions in India, offering an array of travel experiences from the cool Himalayan foothills to the warm beaches of the south. If you’re planning your travel itinerary for November 2026, India presents […]

[Read More](https://www.thomascook.in/blog/india-holidays/12-best-places-to-visit-in-november-2026/)

[![](https://www.thomascook.in/blog/wp-content/plugins/post-carousel/public/assets/img/placeholder.png)](https://www.thomascook.in/blog/forex/what-is-the-average-cost-of-living-in-singapore-in-2026/)

## [What is the average cost of living in Singapore in 2026?](https://www.thomascook.in/blog/forex/what-is-the-average-cost-of-living-in-singapore-in-2026/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 30, 2026

0 0 The Cost of Living in Singapore At A Glance Average monthly budget: A single person may need S$1505 every month excluding accommodation. The rest of the budget depends on lifestyle and location. Biggest expense: Accommodation is the largest […]

[Read More](https://www.thomascook.in/blog/forex/what-is-the-average-cost-of-living-in-singapore-in-2026/)

[![](https://www.thomascook.in/blog/wp-content/plugins/post-carousel/public/assets/img/placeholder.png)](https://www.thomascook.in/blog/forex/top-universities-in-germany-for-international-students/)

## [Top Universities in Germany for International Students](https://www.thomascook.in/blog/forex/top-universities-in-germany-for-international-students/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 30, 2026

0 0 Germany universities for international students have been one of the most preferred destinations for many years. Public universities charge little to no tuition fees, offering excellent opportunities for globally valued research and English-taught programs across engineering, business, and […]

[Read More](https://www.thomascook.in/blog/forex/top-universities-in-germany-for-international-students/)

[![](https://www.thomascook.in/blog/wp-content/plugins/post-carousel/public/assets/img/placeholder.png)](https://www.thomascook.in/blog/forex/rbi-forex-guidelines-for-foreign-exchange-transactions-in-india/)

## [RBI Forex Guidelines for Foreign Exchange Transactions in India](https://www.thomascook.in/blog/forex/rbi-forex-guidelines-for-foreign-exchange-transactions-in-india/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 30, 2026

0 0 What Do RBI Forex Guidelines Say? FEMA oversees the legal mechanism governing foreign exchange transactions in India. All Indian residents should deal in forex only through RBI-authorised persons and for permitted purposes. Under the Liberalised Remittance Scheme (LRS), […]

[Read More](https://www.thomascook.in/blog/forex/rbi-forex-guidelines-for-foreign-exchange-transactions-in-india/)

[![](https://www.thomascook.in/blog/wp-content/plugins/post-carousel/public/assets/img/placeholder.png)](https://www.thomascook.in/blog/forex/cost-of-living-index-by-country-compare-living-expenses-around-the-world-2026-guide/)

## [Cost of Living Index by Country: Compare Living Expenses Around the World (2026 Guide)](https://www.thomascook.in/blog/forex/cost-of-living-index-by-country-compare-living-expenses-around-the-world-2026-guide/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 30, 2026

0 0 No two countries charge the same price for everyday life. A coffee that costs $2 in Hanoi can cost $7 in Zurich, and a one-bedroom apartment renting for $300 in Cairo might command $3,000 in Manhattan. These gaps […]

[Read More](https://www.thomascook.in/blog/forex/cost-of-living-index-by-country-compare-living-expenses-around-the-world-2026-guide/)

[![](https://www.thomascook.in/blog/wp-content/plugins/post-carousel/public/assets/img/placeholder.png)](https://www.thomascook.in/blog/india-holidays/delhi-to-haridwar-distance-best-route-train-timings-travel-guide/)

## [Delhi To Haridwar distance: Best Route, Train Timings & Travel Guide](https://www.thomascook.in/blog/india-holidays/delhi-to-haridwar-distance-best-route-train-timings-travel-guide/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 30, 2026

0 0 Delhi to Haridwar At A Glance Haridwar is an integral religious destination for Hindus. In 2025, 3.4 crore people visited Haridwar, and that number is only expected to be higher at the end of 2026. If you are […]

[Read More](https://www.thomascook.in/blog/india-holidays/delhi-to-haridwar-distance-best-route-train-timings-travel-guide/)

[![](https://www.thomascook.in/blog/wp-content/uploads/2026/07/Kedarnath-Temple-1.webp)](https://www.thomascook.in/blog/india-holidays/delhi-to-kedarnath-distance-best-route-train-timings-travel-guide/)

## [Delhi to Kedarnath distance: Best Route, Train Timings & Travel Guide](https://www.thomascook.in/blog/india-holidays/delhi-to-kedarnath-distance-best-route-train-timings-travel-guide/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 30, 2026

0 0 Delhi to Kedarnath cannot be called a road trip, but it is a journey where the destination is sacred, so every part of the journey becomes a kind of storytelling. Most of the people ask, “Delhi to Kedarnath […]

[Read More](https://www.thomascook.in/blog/india-holidays/delhi-to-kedarnath-distance-best-route-train-timings-travel-guide/)

[![Jodhpur](https://www.thomascook.in/blog/wp-content/uploads/2025/03/5-9.webp)](https://www.thomascook.in/blog/india-holidays/jodhpur-to-udaipur-journey-is-an-ultimate-rajasthan-travel-journey/)

## [Jodhpur to Udaipur journey is an ultimate Rajasthan travel journey.](https://www.thomascook.in/blog/india-holidays/jodhpur-to-udaipur-journey-is-an-ultimate-rajasthan-travel-journey/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 30, 2026

0 0 These two destinations have completely different personalities, but the journey between the two destinations is mesmerising. Imagine yourself waking up to the blue-painted buildings of Jodhpur, covered with mighty forts and surrounded by beautiful desert vibes. And then […]

[Read More](https://www.thomascook.in/blog/india-holidays/jodhpur-to-udaipur-journey-is-an-ultimate-rajasthan-travel-journey/)

[![Manali ](https://www.thomascook.in/blog/wp-content/uploads/2025/03/2-5.webp)](https://www.thomascook.in/blog/india-holidays/kullu-to-manali-distance-best-route-train-timings-road-trip-guide/)

## [Kullu to Manali distance: Best Route, Train Timings & Road Trip Guide](https://www.thomascook.in/blog/india-holidays/kullu-to-manali-distance-best-route-train-timings-road-trip-guide/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 30, 2026

0 0 Kullu to Manali At a Glance Kullu and Manali sit in the same valley, so this isn’t really a “journey” in the way Delhi to Manali is; it’s more of a short hop most travellers do on their […]

[Read More](https://www.thomascook.in/blog/india-holidays/kullu-to-manali-distance-best-route-train-timings-road-trip-guide/)

[![](https://www.thomascook.in/blog/wp-content/uploads/2023/11/mumbai-coastline-.webp)](https://www.thomascook.in/blog/india-holidays/everything-you-need-to-know-about-mumbai-to-goa-distance/)

## [Everything You Need to Know About Mumbai to Goa Distance](https://www.thomascook.in/blog/india-holidays/everything-you-need-to-know-about-mumbai-to-goa-distance/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 30, 2026

0 0 Mumbai to Goa At a Glance Goa is one of the most happening places in India. It offers nature, partying, waterfalls, trails, water activities, and whatnot. Goa sees millions of tourists every year from all across the globe, […]

[Read More](https://www.thomascook.in/blog/india-holidays/everything-you-need-to-know-about-mumbai-to-goa-distance/)

[![Varanasi Ghat](https://www.thomascook.in/blog/wp-content/uploads/2024/06/Varanasi-Ghat.webp)](https://www.thomascook.in/blog/india-holidays/varanasi-to-haridwar-distance-best-route-train-timings-travel-guide/)

## [Varanasi to Haridwar distance: Best Route, Train Timings & Travel Guide](https://www.thomascook.in/blog/india-holidays/varanasi-to-haridwar-distance-best-route-train-timings-travel-guide/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 30, 2026

0 0 Varanasi to Haridwar at a Glance Here are the numbers, before anything else. People do this route mostly for one reason: both cities run a daily Ganga aarti, and pilgrims like pairing them into one circuit. Road: roughly […]

[Read More](https://www.thomascook.in/blog/india-holidays/varanasi-to-haridwar-distance-best-route-train-timings-travel-guide/)

[![](https://www.thomascook.in/blog/wp-content/plugins/post-carousel/public/assets/img/placeholder.png)](https://www.thomascook.in/blog/international-holidays/ride-the-worlds-longest-cable-car-to-hon-thom-island-your-complete-phu-quoc-travel-guide/)

## [Ride the World’s Longest Cable Car to Hon Thom Island: Your Complete Phu Quoc Travel Guide](https://www.thomascook.in/blog/international-holidays/ride-the-worlds-longest-cable-car-to-hon-thom-island-your-complete-phu-quoc-travel-guide/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 29, 2026

0 0 The Hon Thom cable car in Phu Quoc is renowned for its Guinness World Record as the longest sea-crossing cable car. Offering a great view of the Gulf of Thailand, this cable car ride stretches across 7.9 km. […]

[Read More](https://www.thomascook.in/blog/international-holidays/ride-the-worlds-longest-cable-car-to-hon-thom-island-your-complete-phu-quoc-travel-guide/)

[![](https://www.thomascook.in/blog/wp-content/plugins/post-carousel/public/assets/img/placeholder.png)](https://www.thomascook.in/blog/international-holidays/longest-bridge-in-the-world-where-record-breaking-engineering-meets-stunning-landscapes/)

## [Longest Bridge in the World: Where Record-Breaking Engineering Meets Stunning Landscapes](https://www.thomascook.in/blog/international-holidays/longest-bridge-in-the-world-where-record-breaking-engineering-meets-stunning-landscapes/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 29, 2026

0 0 Ask most people to name the world’s longest bridge, and you’re likely to get a confident answer like the Golden Gate Bridge or the Brooklyn Bridge or some other famous landmark plastered across postcards and movies. But the […]

[Read More](https://www.thomascook.in/blog/international-holidays/longest-bridge-in-the-world-where-record-breaking-engineering-meets-stunning-landscapes/)

[![](https://www.thomascook.in/blog/wp-content/plugins/post-carousel/public/assets/img/placeholder.png)](https://www.thomascook.in/blog/international-holidays/worlds-largest-waterfall-experience-the-majestic-beauty-of-angel-falls-like-never-before/)

## [World’s Largest Waterfall: Experience the Majestic Beauty of Angel Falls Like Never Before](https://www.thomascook.in/blog/international-holidays/worlds-largest-waterfall-experience-the-majestic-beauty-of-angel-falls-like-never-before/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 28, 2026

0 0 Angel Falls in Venezuela is the highest waterfall in the world. It is 979 metres high. The falls are in a large forest area called Canaima National Park. Getting there takes time, but the views are amazing. You […]

[Read More](https://www.thomascook.in/blog/international-holidays/worlds-largest-waterfall-experience-the-majestic-beauty-of-angel-falls-like-never-before/)

[![](https://www.thomascook.in/blog/wp-content/plugins/post-carousel/public/assets/img/placeholder.png)](https://www.thomascook.in/blog/international-holidays/visit-the-largest-mall-in-the-world-everything-you-need-for-the-perfect-shopping-trip/)

## [Visit the Largest Mall in the World: Everything You Need for the Perfect Shopping Trip](https://www.thomascook.in/blog/international-holidays/visit-the-largest-mall-in-the-world-everything-you-need-for-the-perfect-shopping-trip/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 28, 2026

0 0 Largest Mall In The World Iran Mall in Tehran, Iran, is the world’s biggest mall by total reported area. The mall is approximately 1.95 million square metres, or 21 million square feet. Note: The discussion of the biggest […]

[Read More](https://www.thomascook.in/blog/international-holidays/visit-the-largest-mall-in-the-world-everything-you-need-for-the-perfect-shopping-trip/)

[![](https://www.thomascook.in/blog/wp-content/plugins/post-carousel/public/assets/img/placeholder.png)](https://www.thomascook.in/blog/international-holidays/top-10-highest-railway-bridges-in-the-world-discover-the-tallest-train-bridges-and-their-incredible-stories/)

## [Top 10 Highest Railway Bridges in the World: Discover the Tallest Train Bridges and Their Incredible Stories](https://www.thomascook.in/blog/international-holidays/top-10-highest-railway-bridges-in-the-world-discover-the-tallest-train-bridges-and-their-incredible-stories/)

* [Thomas Cook Admin](https://www.thomascook.in/blog/author/dev_admin/)
* September 28, 2026

0 0  The world’s highest railway bridges are mostly in China and India. India’s Chenab Bridge is the tallest railway arch bridge at 359 metres, which is even higher than the Eiffel Tower. China’s Najiehe Bridge, at 310 metres, still […]

[Read More](https://www.thomascook.in/blog/international-holidays/top-10-highest-railway-bridges-in-the-world-discover-the-tallest-train-bridges-and-their-incredible-stories/)

##### International Tour Packages

* [International Tour Packages](https://www.thomascook.in/holidays/international-tour-packages "International Tour Packages")
* [Dubai Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/dubai-tour-packages "Dubai Tour Packages")
* [Europe Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/europe-tour-packages "Europe Tour Packages")
* [Thailand Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/thailand-tour-packages "Thailand Tour Packages")
* [Bali Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/bali-tour-packages "Bali Tour Packages")
* [Vietnam Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/vietnam-tour-packages "Vietnam Tour Packages")
* [Singapore Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/singapore-tour-packages "Singapore Tour Packages")
* [Japan Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/japan-tour-packages "Japan Tour Packages")
* [Maldives Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/maldives-tour-packages "Maldives Tour Packages")
* [Switzerland Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/switzerland-tour-packages "Switzerland Tour Packages")
* [Malaysia Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/malaysia-tour-packages "Malaysia Tour Packages")
* [USA Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/usa-america-tour-packages "USA Tour Packages")
* [Mauritius Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/mauritius-tour-packages "Mauritius Tour Packages")
* [Australia Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/australia-tour-packages "Australia Tour Packages")
* [Turkey Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/turkey-tour-packages "Turkey Tour Packages")
* [Nepal Tour Packages](https://www.thomascook.in/holidays/international-tour-packages/nepal-tour-packages "Nepal Tour Packages")

##### India Tour Packages

* [India Tour Packages](https://www.thomascook.in/holidays/india-tour-packages "India Tour Packages")
* [Kashmir Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/jammu-and-kashmir-tour-packages "Kashmir Tour Packages")
* [Cordelia Cruise Packages](https://www.thomascook.in/holidays/india-tour-packages/cordelia-cruise-tour-packages "Cordelia Cruise Packages")
* [Kerala Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/kerala-tour-packages "Kerala Tour Packages")
* [Sri Lanka Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/sri-lanka-tour-packages "Sri Lanka Tour Packages")
* [Andaman Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/andaman-and-nicobar-tour-packages "Andaman Tour Packages")
* [Bhutan Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/bhutan-tour-packages "Bhutan Tour Packages")
* [Goa Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/goa-tour-packages "Goa Tour Packages")
* [Himachal Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/himachal-pradesh-tour-packages "Himachal Tour Packages")
* [Lakshadweep Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/lakshadweep-tour-packages "Lakshadweep Tour Packages")
* [Ladakh Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/ladakh-tour-packages "Ladakh Tour Packages")
* [Rajasthan Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/rajasthan-tour-packages "Rajasthan Tour Packages")
* [North East Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/north-east-tour-packages "North East Tour Packages")
* [Uttarakhand Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/uttarakhand-tour-packages "Uttarakhand Tour Packages")
* [Manali Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/manali-tour-packages "Manali Tour Packages")
* [Sikkim Tour Packages](https://www.thomascook.in/holidays/india-tour-packages/sikkim-tour-packages "Sikkim Tour Packages")

##### Honeymoon Packages

* [International Honeymoon Packages](https://www.thomascook.in/holidays/international-honeymoon-packages "International Honeymoon Packages")
* [Bali Honeymoon Packages](https://www.thomascook.in/holidays/international-honeymoon-packages/bali-honeymoon-packages "Bali Honeymoon Packages")
* [Maldives Honeymoon Packages](https://www.thomascook.in/holidays/international-honeymoon-packages/maldives-honeymoon-packages "Maldives Honeymoon Packages")
* [Switzerland Honeymoon Packages](https://www.thomascook.in/holidays/international-honeymoon-packages/switzerland-honeymoon-packages "Switzerland Honeymoon Packages")
* [Europe Honeymoon Packages](https://www.thomascook.in/holidays/international-honeymoon-packages/europe-honeymoon-packages "Europe Honeymoon Packages")
* [Dubai Honeymoon Packages](https://www.thomascook.in/holidays/international-honeymoon-packages/dubai-honeymoon-packages "Dubai Honeymoon Packages")
* [Thailand Honeymoon Packages](https://www.thomascook.in/holidays/international-honeymoon-packages/thailand-honeymoon-packages "Thailand Honeymoon Packages")
* [Singapore Honeymoon Packages](https://www.thomascook.in/holidays/international-honeymoon-packages/singapore-honeymoon-packages "Singapore Honeymoon Packages")
* [India Honeymoon Packages](https://www.thomascook.in/holidays/honeymoon-packages-india "India Honeymoon Packages")
* [Goa Honeymoon Packages](https://www.thomascook.in/holidays/honeymoon-packages-india/goa-honeymoon-packages "Goa Honeymoon Packages")
* [Kashmir Honeymoon Packages](https://www.thomascook.in/holidays/honeymoon-packages-india/jammu-and-kashmir-honeymoon-packages "Kashmir Honeymoon Packages")
* [Kerala Honeymoon Packages](https://www.thomascook.in/holidays/honeymoon-packages-india/kerala-honeymoon-packages "Kerala Honeymoon Packages")
* [Himachal Honeymoon Packages](https://www.thomascook.in/holidays/honeymoon-packages-india/himachal-pradesh-honeymoon-packages "Himachal Honeymoon Packages")
* [Manali Honeymoon Packages](https://www.thomascook.in/holidays/honeymoon-packages-india/manali-honeymoon-packages "Manali Honeymoon Packages")
* [Andaman Honeymoon Packages](https://www.thomascook.in/holidays/honeymoon-packages-india/andaman-and-nicobar-honeymoon-packages "Andaman Honeymoon Packages")
* [Sri Lanka Honeymoon Packages](https://www.thomascook.in/holidays/honeymoon-packages-india/sri-lanka-honeymoon-packages "Sri Lanka Honeymoon Packages")

##### Foreign Exchange

* [Foreign Exchange](https://www.thomascook.in/foreign-exchange "Foreign Exchange")
* [Buy Forex](https://www.thomascook.in/foreign-exchange/buy-forex-online "Buy Forex")
* [Forex Cards](https://www.thomascook.in/foreign-exchange/travel-prepaid-forex-cards "Forex Cards")
* [Multi Currency Travel Card](https://www.thomascook.in/foreign-exchange/borderless-prepaid-multicurrency-card "Multi Currency Travel Card")
* [Buy Cash Currency](https://www.thomascook.in/foreign-exchange/buy-cash-currency "Buy Cash Currency")
* [Forex Rate Card](https://www.thomascook.in/foreign-exchange/forex-rate-card "Forex Rate Card")
* [One Currency Card](https://www.thomascook.in/foreign-exchange/one-currency-card "One Currency Card")
* [International Money Transfer](https://www.thomascook.in/international-money-transfer "International Money Transfer")
* [Remit From India](https://www.thomascook.in/remit-from-india "Remit From India")
* [Reload Forex Card](https://www.thomascook.in/foreign-exchange/reload-forex-card "Reload Forex Card")
* [Sell Forex](https://www.thomascook.in/foreign-exchange/sell-forex-online "Sell Forex")

##### Travel News

* [Travel News](https://www.thomascook.in/blog/travel-news/ "Travel News")

###### © COPYRIGHT THOMASCOOK 2026.

![](https://blogs.thomascook.in//wp-content/uploads/2023/08/TC_logo-removebg-preview.png)

## Discover Your Next Journey With Thomas Cook

## Book Your Next Holiday and Get the Best Deals. Inquire Now!

Select ProductInternational HolidaysDomestic HolidaysForex

## Discover Your Next Journey With Thomas Cook

## Book Your Next Holiday and Get the Best Deals. Inquire Now!

Select ProductInternational HolidaysDomestic HolidaysForex

## Discover Your Next Journey With Thomas Cook

## Book Your Next Holiday and Get the Best Deals. Inquire Now!

Select ProductInternational HolidaysDomestic HolidaysForex

## Discover Your Next Journey With Thomas Cook

## Book Your Next Holiday and Get the Best Deals. Inquire Now!

Select ProductInternational HolidaysDomestic HolidaysForex

## Discover Your Next Journey With Thomas Cook

## Book Your Next Holiday and Get the Best Deals. Inquire Now!

Select ProductInternational HolidaysDomestic HolidaysForex

CLOSE

In [25]:
@tool
def get_weather(query: str) -> list:
  """ search the weather API to get the current weather for queried loataion ."""
  print("calling wather tool")
  base_url = "http://api.weatherapi.com/v1/current.json"
  complete_url = f"{base_url}?key={WEATHER_API_KEY}&q={query}"


  response =  request.get(complete_url)
  data = response.json()
  if data.get("location"):
      return data
  else:
     return("weather data not found")